"""feature_engineering.ipynb
Tạo các đặc trưng (features) cho bài toán dự đoán churn.
- Đầu vào: Dữ liệu đã được làm sạch và gộp (merged_orders.parquet)
- Đầu ra: Bảng features ở cấp độ khách hàng (customer_unique_id)
"""

In [1]:
import pandas as pd
import numpy as np
from churn_prediction.paths import PROCESSED_DIR, FEATURES_DIR, INTERIM_CLI_DIR
import warnings
warnings.filterwarnings('ignore')

All directories created successfully!


In [2]:
# Dữ liệu đã được gom nhóm theo customer
customer_df = pd.read_parquet(PROCESSED_DIR / 'customer_level.parquet')

# Dữ liệu gốc để tính các features
df_merged = pd.read_parquet(INTERIM_CLI_DIR / 'merged_orders.parquet')

print(f"   Customer level: {customer_df.shape}")
print(f"   Merged orders: {df_merged.shape}")

   Customer level: (96096, 32)
   Merged orders: (99441, 26)


In [3]:
# Dữ liệu đã được gom nhóm theo customer
customer_df = pd.read_parquet(PROCESSED_DIR / 'customer_level.parquet')

# Dữ liệu gốc để tính các features
df_merged = pd.read_parquet(INTERIM_CLI_DIR / 'merged_orders.parquet')

print(f"   Customer level: {customer_df.shape}")
print(f"   Merged orders: {df_merged.shape}")

   Customer level: (96096, 32)
   Merged orders: (99441, 26)


### NHÓM 1: Trải nhiệm giao hàng (logistics)

In [4]:
# Tính thời gian giao hàng (đã có trong merged orders)
# delivery_days = ngày giao hàng thực tế - ngày mua
df_merged['delivery_time_days'] = (
    df_merged['order_delivered_customer_date'] - df_merged['order_purchase_timestamp']
).dt.days

# Thời gian giao hàng dự kiến
df_merged['estimated_delivery_days'] = (
    df_merged['order_estimated_delivery_date'] - df_merged['order_purchase_timestamp']
).dt.days

# Độ trễ giao hàng (ngày giao trễ so với dự kiến)
df_merged['delivery_delay'] = df_merged['delivery_time_days'] - df_merged['estimated_delivery_days']

# Aggregation theo customer
delivery_features = df_merged.groupby('customer_unique_id').agg({
    'delivery_time_days': ['mean', 'std', 'max'],
    'estimated_delivery_days': 'mean',
    'delivery_delay': ['mean', 'std', 'max'],
    'avg_freight': ['mean', 'sum'], 
    'total_freight': 'mean',
    
}).reset_index()

delivery_features.columns = [
    'customer_unique_id',
    'avg_delivery_time', 'std_delivery_time', 'max_delivery_time',
    'avg_estimated_delivery',
    'avg_delivery_delay', 'std_delivery_delay', 'max_delivery_delay',
    'avg_freight_per_order', 'total_freight_from_avg',
    'total_freight_all'
]

# Tỉ lệ giao hàng trễ (late delivery ratio)
temp = df_merged.groupby('customer_unique_id').apply(
    lambda x: (x['delivery_delay'] > 0).mean()
).reset_index()
temp.columns = ['customer_unique_id', 'late_delivery_ratio']
delivery_features = delivery_features.merge(temp, on='customer_unique_id', how='left')
logistics = delivery_features
print(f"   Đã tạo {len(logistics.columns)-1} logistics features")

   Đã tạo 11 logistics features


### NHÓM 2: Review

In [5]:
print("\n3. Nhóm 2: Review features")
review = df_merged.groupby('customer_unique_id').agg({
    'review_score': ['mean', 'std', 'min'],
    'num_comment_messages': lambda x: (~x.isna()).sum(),
    'num_comment_titles': lambda x: (~x.isna()).sum(),
    'days_to_answer': 'mean'
}).reset_index()
review.columns = ['customer_unique_id',
                  'avg_review_score', 'std_review_score', 'min_review_score',
                  'num_comments', 'num_titles', 'avg_days_to_answer']

low_review = df_merged.groupby('customer_unique_id')['review_score'].apply(lambda x: (x <= 2).mean())
review = review.merge(low_review.rename('low_review_ratio'), on='customer_unique_id')
bad_review_count = df_merged.groupby('customer_unique_id')['review_score'].apply(lambda x: (x <= 2).sum())
review = review.merge(bad_review_count.rename('num_bad_reviews'), on='customer_unique_id')
print(f"   -> {review.shape[1]-1} features")


3. Nhóm 2: Review features
   -> 8 features


### NHÓM 3: Hành vi

In [6]:
print("\n4. Nhóm 3: Behavior features")
df_merged['purchase_hour'] = df_merged['order_purchase_timestamp'].dt.hour
df_merged['purchase_weekday'] = df_merged['order_purchase_timestamp'].dt.dayofweek
df_merged['is_weekend'] = df_merged['purchase_weekday'].isin([5,6]).astype(int)
df_merged['is_night'] = df_merged['purchase_hour'].isin(range(20,24)).astype(int)

behavior = df_merged.groupby('customer_unique_id').agg({
    'num_products': ['mean', 'std'],
    'is_weekend': 'mean',
    'is_night': 'mean',
    'order_id': 'count'
}).reset_index()
behavior.columns = ['customer_unique_id',
                    'avg_items_per_order', 'std_items_per_order',
                    'weekend_purchase_ratio', 'night_purchase_ratio',
                    'total_orders']

# Khoảng cách giữa các lần mua
order_times = df_merged.sort_values(['customer_unique_id', 'order_purchase_timestamp'])
def get_gaps(group):
    if len(group) < 2:
        return pd.Series({'avg_gap': 0, 'std_gap': 0, 'max_gap': 0, 'trend_gap': 0})
    gaps = group.diff().dt.days.dropna()
    trend = np.polyfit(range(len(gaps)), gaps, 1)[0] if len(gaps) > 1 else 0
    return pd.Series({'avg_gap': gaps.mean(), 'std_gap': gaps.std(), 'max_gap': gaps.max(), 'trend_gap': trend})
gap_features = order_times.groupby('customer_unique_id')['order_purchase_timestamp'].apply(get_gaps).unstack().reset_index()
behavior = behavior.merge(gap_features, on='customer_unique_id')
print(f"   -> {behavior.shape[1]-1} features")


4. Nhóm 3: Behavior features
   -> 9 features


### NHÓM 4: RFM

In [7]:
print("\n5. Nhóm 4: RFM features")
rfm = df_merged.groupby('customer_unique_id').agg({
    'order_purchase_timestamp': lambda x: (pd.Timestamp.now() - x.max()).days,
    'order_id': 'count',
    'total_payment': 'sum'
}).reset_index()
rfm.columns = ['customer_unique_id', 'recency', 'frequency', 'monetary']

rfm['recency_score'] = pd.qcut(rfm['recency'].rank(method='first'), q=4, labels=[4,3,2,1]).astype(int)
rfm['frequency_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=4, labels=[1,2,3,4]).astype(int)
rfm['monetary_score'] = pd.qcut(rfm['monetary'].rank(method='first'), q=4, labels=[1,2,3,4]).astype(int)
rfm['rfm_total'] = rfm['recency_score'] + rfm['frequency_score'] + rfm['monetary_score']

def rfm_segment(row):
    if row['recency_score']>=3 and row['frequency_score']>=3 and row['monetary_score']>=3:
        return 'champion'
    if row['recency_score']>=3 and row['frequency_score']>=2:
        return 'loyal'
    if row['recency_score']<=2 and row['frequency_score']<=2 and row['monetary_score']<=2:
        return 'at_risk'
    if row['recency_score']<=1:
        return 'churned'
    return 'promising'
rfm['rfm_segment'] = rfm.apply(rfm_segment, axis=1)
print(f"   -> {rfm.shape[1]-1} features")


5. Nhóm 4: RFM features
   -> 8 features


### NHÓM 5: Thanh toán 

In [8]:
print("\n6. Nhóm 5: Payment features")
payment = df_merged.groupby('customer_unique_id').agg({
    'max_installments': 'mean',
    'main_payment_type': lambda x: x.mode()[0] if len(x) > 0 else 'unknown'
}).reset_index()
payment.columns = ['customer_unique_id', 'avg_installments', 'favorite_payment_type']

credit_ratio = df_merged.groupby('customer_unique_id')['main_payment_type'].apply(lambda x: (x == 'credit_card').mean())
boleto_ratio = df_merged.groupby('customer_unique_id')['main_payment_type'].apply(lambda x: (x == 'boleto').mean())
payment = payment.merge(credit_ratio.rename('credit_card_ratio'), on='customer_unique_id')
payment = payment.merge(boleto_ratio.rename('boleto_ratio'), on='customer_unique_id')
print(f"   -> {payment.shape[1]-1} features")


6. Nhóm 5: Payment features
   -> 4 features


### NHÓM 6: Xu hướng

In [9]:
print("\n7. Nhóm 6: Trend features")
df_sorted = df_merged.sort_values(['customer_unique_id', 'order_purchase_timestamp'])
def trend_features(group):
    if len(group) < 3:
        return pd.Series({'spending_trend':0, 'order_trend':0, 'recent_vs_old_ratio':0, 'avg_order_trend':0})
    n = len(group)
    half = n // 2
    old = group.head(half)
    recent = group.tail(half)
    old_spend = old['total_payment'].mean()
    recent_spend = recent['total_payment'].mean()
    spending_trend = (recent_spend - old_spend) / (old_spend + 1)
    order_trend = (len(recent) - len(old)) / (len(old) + 1)
    recent_vs_old_ratio = len(recent) / (len(old) + 1)
    old_avg = old['total_payment'].sum()/len(old)
    recent_avg = recent['total_payment'].sum()/len(recent)
    avg_order_trend = (recent_avg - old_avg) / (old_avg + 1)
    return pd.Series({'spending_trend':spending_trend, 'order_trend':order_trend,
                      'recent_vs_old_ratio':recent_vs_old_ratio, 'avg_order_trend':avg_order_trend})
trend = df_sorted.groupby('customer_unique_id').apply(trend_features).reset_index()
print(f"   -> {trend.shape[1]-1} features")


7. Nhóm 6: Trend features
   -> 4 features


### NHÓM 7: Tương Tác

In [10]:
print(df_merged.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'total_price', 'avg_price', 'num_products', 'total_freight', 'avg_freight', 'unique_products', 'unique_sellers', 'total_payment', 'max_installments', 'main_payment_type', 'review_score', 'num_comment_messages', 'num_comment_titles', 'days_to_answer', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'delivery_time_days', 'estimated_delivery_days', 'delivery_delay', 'purchase_hour', 'purchase_weekday', 'is_weekend', 'is_night']


In [12]:
print("\n8. Nhóm 7: Interaction features")

interaction = df_merged.groupby('customer_unique_id').agg({
    'num_comment_messages': 'sum',
    'num_comment_titles': 'sum',
    'days_to_answer': 'mean',
    'unique_sellers': 'sum'
}).reset_index()

interaction.columns = ['customer_unique_id', 'total_comments_msg', 'total_comments_title', 
                       'avg_days_to_answer', 'unique_sellers']

# Lấy total_orders từ behavior (đã được tính ở nhóm 3)
total_orders = behavior[['customer_unique_id', 'total_orders']].copy()

# Merge để có total_orders
interaction = interaction.merge(total_orders, on='customer_unique_id', how='left')

# Tính comment_rate
interaction['comment_rate'] = interaction['total_comments_msg'] / (interaction['total_orders'] + 1)

# (Tùy chọn) Bỏ cột total_orders nếu không cần
interaction = interaction.drop('total_orders', axis=1)

print(f"   -> {interaction.shape[1]-1} features")


8. Nhóm 7: Interaction features
   -> 5 features


### NHÓM 8: Địa Lý

In [13]:
print("\n9. Nhóm 8: Geo features")
geo = df_merged[['customer_unique_id', 'customer_state', 'customer_city']].drop_duplicates()
print(f"   -> {geo.shape[1]-1} features")


9. Nhóm 8: Geo features
   -> 2 features


### NHÓM 9: Sản Phẩm

In [14]:
print("\n10. Nhóm 9: Product features")

# Đọc các bảng cần thiết
order_items = pd.read_parquet(PROCESSED_DIR / 'order_items_clean.parquet')
products = pd.read_parquet(PROCESSED_DIR / 'products_clean.parquet')
orders = pd.read_parquet(PROCESSED_DIR / 'orders_clean.parquet')
customers = pd.read_parquet(PROCESSED_DIR / 'customers_clean.parquet')

# Gắn category
df_prod = order_items.merge(products[['product_id', 'product_category_name']], on='product_id', how='left')

# Gắn customer_id từ orders
df_prod = df_prod.merge(orders[['order_id', 'customer_id']], on='order_id', how='left')

# Gắn customer_unique_id từ customers
df_prod = df_prod.merge(customers[['customer_id', 'customer_unique_id']], on='customer_id', how='left')

# Tính favourite category (mode)
favorite_cat = df_prod.groupby('customer_unique_id')['product_category_name'].agg(
    lambda x: x.mode()[0] if len(x.mode()) > 0 else 'unknown'
)

# Tính số lượng category đã mua
num_categories = df_prod.groupby('customer_unique_id')['product_category_name'].nunique()

# Tạo dataframe product features
product = pd.DataFrame({
    'customer_unique_id': favorite_cat.index,
    'favorite_category': favorite_cat.values
})
product = product.merge(num_categories.rename('num_categories_bought'), on='customer_unique_id')

print(f"   -> {product.shape[1]-1} features")


10. Nhóm 9: Product features
   -> 2 features


### 11. Gộp tất cả

In [15]:
# Kiểm tra số lượng unique customer trong mỗi bảng
print("\nKiểm tra số lượng khách hàng trong các bảng intermediate:")
print(f"rfm: {rfm['customer_unique_id'].nunique()}")
print(f"logistics: {logistics['customer_unique_id'].nunique()}")
print(f"review: {review['customer_unique_id'].nunique()}")
print(f"behavior: {behavior['customer_unique_id'].nunique()}")
print(f"payment: {payment['customer_unique_id'].nunique()}")
print(f"trend: {trend['customer_unique_id'].nunique()}")
print(f"interaction: {interaction['customer_unique_id'].nunique()}")
print(f"geo: {geo['customer_unique_id'].nunique()}")
print(f"product: {product['customer_unique_id'].nunique()}")


Kiểm tra số lượng khách hàng trong các bảng intermediate:
rfm: 96096
logistics: 96096
review: 96096
behavior: 96096
payment: 96096
trend: 96096
interaction: 96096
geo: 96096
product: 94983


In [16]:
print("\n11. Gộp các nhóm features...")
final_features = rfm
for df in [logistics, review, behavior, payment, trend, interaction, geo, product]:
    final_features = final_features.merge(df, on='customer_unique_id', how='left')

# Xử lý missing và infinite
final_features = final_features.replace([np.inf, -np.inf], 0)
for col in final_features.select_dtypes(include=[np.number]).columns:
    final_features[col] = final_features[col].fillna(0)
for col in final_features.select_dtypes(include=['object']).columns:
    if col != 'customer_unique_id':
        final_features[col] = final_features[col].fillna('unknown')

print(f"   Tổng số features (không kể ID): {final_features.shape[1]-1}")
print(f"   Kích thước bảng features: {final_features.shape}")


11. Gộp các nhóm features...
   Tổng số features (không kể ID): 53
   Kích thước bảng features: (96219, 54)


### 12. Lưu features

In [17]:
print("\n12. Lưu features...")
final_features.to_parquet(FEATURES_DIR / 'customer_features.parquet', index=False)
print(f"Đã lưu customer_features.parquet")

feature_names = [col for col in final_features.columns if col != 'customer_unique_id']
with open(FEATURES_DIR / 'feature_names.txt', 'w') as f:
    f.write('\n'.join(feature_names))

print("\n=== FEATURE ENGINEERING HOÀN TẤT ===")


12. Lưu features...
Đã lưu customer_features.parquet

=== FEATURE ENGINEERING HOÀN TẤT ===
